In [11]:
# імпортуємо бібліотеку   pandas
import pandas as pd

In [12]:
# зчитуємо CSV-файли в Pandas і завантажуємо в таблиці df  і df1
df = pd.read_csv(r'D:\GO_IT\Проєкт2\survey_results_public.csv', encoding='UTF-8')
df1 = pd.read_csv(r'D:\GO_IT\Проєкт2\survey_results_schema.csv', encoding='UTF-8')
#df.shape[0]

C:\Users\M&S\AppData\Local\Temp\ipykernel_17920\3432534420.py:2: DtypeWarning: Columns (56,74,92,97,98,105,109,110,132,162,165) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'D:\GO_IT\Проєкт2\survey_results_public.csv', encoding='UTF-8')


In [13]:
#налаштування параметрів для відображення всіх колонок
pd.set_option('display.max_column',None)

In [14]:
# Завдання 1. Підрахунок загальної кількості респондентів
respondents_count = df['ResponseId'].nunique()
respondents_count

49191

In [6]:
# df.shape[0] == df['ResponseId'].nunique()
# немає дублікатів:

In [15]:
#Завдання 2. Аналіз повноти відповідей респондентів
# Створюємо множину назв питань з колонки 'qname' (уникальні)
questions_set = set(df1['qname'].dropna().unique())
# Обираємо лише ті питання, які реально є у нашому датафреймі
existing_questions = questions_set.intersection(df.columns)
# Видаляємо рядки, де є пропуски у цих колонках
complete_respondents = df[list(existing_questions)].dropna()
# Кількість респондентів, які відповіли на всі питання
num_complete_respondents = complete_respondents.shape[0]
num_complete_respondents

0

In [16]:
#№Завдання 3. Статистичний аналіз досвіду респондентів
# Вибираємо колонку з досвідом роботи
work_exp = df['WorkExp']
# Обчислюємо статистичні показники, пропускаючи пропущені значення  NaN
mean_exp = work_exp.mean(skipna=True).round(2)
median_exp = work_exp.median(skipna=True)
mode_exp = work_exp.mode(dropna=True)
# Виводимо результат
print(f"Міри центральної тенденції для досвіду роботи респондентів:")
print(f"Середнє значення (mean): {mean_exp}")
print(f"Медіана (median): {median_exp}")
print(f"Мода (mode): {list(mode_exp)}")


Міри центральної тенденції для досвіду роботи респондентів:
Середнє значення (mean): 13.37
Медіана (median): 10.0
Мода (mode): [10.0]


In [17]:
#Завдання 4. Аналіз віддаленої роботи
# Підрахунок респондентів, які працюють віддалено (Remote)
remote_count = (df['RemoteWork'] == 'Remote').sum()
print(remote_count)

10931


In [18]:
#Завдання 5. Визначення популярності Python
# Створюємо копію датафрейму
df_copy = df.copy()
# Створюємо новий стовпець Worked with Python
df_copy["Worked with Python"] = df_copy["LanguageHaveWorkedWith"] \
    .str.contains("Python", case=False, na=False)
# Кількість респондентів, які працювали з Python
python_count = df_copy["Worked with Python"].sum()
# Загальна кількість респондентів
total_count = len(df_copy)
# Відсоток респондентів, які програмують на Python
ВідсотокРеспондентів = python_count / total_count * 100
print(f"{ВідсотокРеспондентів:.1f}%")

37.5%


In [19]:
#Завдання 6. Аналіз шляхів навчання програмуванню
# Працюємо з копією датафрейму
df_copy = df.copy()
# Створюємо новий стовпець: чи навчався респондент через онлайн-курси
df_copy["Learned with online courses"] = df_copy["LearnCode"] \
    .str.contains("Online courses", case=False, na=False)
# Підраховуємо кількість таких респондентів
online_courses_count = df_copy["Learned with online courses"].sum()
print(f"Кількість респондентів, які навчались через онлайн-курси: {online_courses_count}")


Кількість респондентів, які навчались через онлайн-курси: 10973


In [21]:
#Завдання 7. Географічний аналіз компенсації Python-розробників

df_copy = df.copy()
# Відбір респондентів, які працювали з Python
python_devs = df_copy[
    df_copy["LanguageHaveWorkedWith"]
    .str.contains("Python", case=False, na=False)
]
# Видаляємо рядки без інформації про компенсацію
python_devs = python_devs.dropna(subset=["ConvertedCompYearly"])
# Групування за країнами та розрахунок статистик
salary_by_country = (
    python_devs
    .groupby("Country")["ConvertedCompYearly"]
    .agg(
        avg_compensation="mean",
        median_compensation="median",
        respondents_count="count"   # додатково: кількість Python-розробників у країні
    )
    .reset_index()
)
# Округлення значень для кращої читабельності
salary_by_country["avg_compensation"] = salary_by_country["avg_compensation"].round(2)
salary_by_country["median_compensation"] = salary_by_country["median_compensation"].round(2)
# Сортування за середньою компенсацією (за спаданням)
salary_by_country = salary_by_country.sort_values(by="avg_compensation", ascending=False)
salary_by_country


,Country,avg_compensation,median_compensation,respondents_count
102,Oman,390135.00,390135.0,2
3,Andorra,226103.50,226103.5,2
149,Viet Nam,218837.17,8254.0,30
145,United States of America,173298.59,150000.0,3126
132,Switzerland,156456.60,142592.0,240
...,...,...,...,...
20,Botswana,1277.00,1277.0,1
24,Cambodia,1270.00,1270.0,1
136,Togo,354.00,354.0,1
104,Palestine,78.00,78.0,1


In [22]:
#Завдання 8. Аналіз освіти найбільш оплачуваних спеціалістів
# Працюємо з копією датафрейму
df_copy = df.copy()
# бираємо потрібні колонки
cols = ["ResponseId", "EdLevel", "ConvertedCompYearly"]
df_selected = df_copy[cols]
# Видаляємо рядки без значення зарплати
df_selected = df_selected.dropna(subset=["ConvertedCompYearly"])
# Сортуємо за компенсацією (за спаданням)
df_sorted = df_selected.sort_values(by="ConvertedCompYearly", ascending=False)
# Оновлюємо індекс
df_sorted = df_sorted.reset_index(drop=True)
# Обираємо топ-5 респондентів
top5_education = df_sorted.head(5)
# Виводимо результат
top5_education

,ResponseId,EdLevel,ConvertedCompYearly
0,34268,"Associate degree (A.A., A.S., etc.)",50000000.0
1,28701,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",33552715.0
2,43144,"Associate degree (A.A., A.S., etc.)",18387548.0
3,35354,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",15430267.0
4,45972,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",13921760.0


In [23]:
#Завдання 9. Аналіз популярності Python по віковим категоріям 

df_copy = df.copy()
# Створюємо колонку: чи працював респондент з Python
df_copy["PythonUser"] = (
    df_copy["LanguageHaveWorkedWith"]
    .str.contains("Python", case=False, na=False)
)
# Групування по вікових категоріях
age_python_stats = (
    df_copy
    .groupby("Age")["PythonUser"]
    .mean() * 100   # mean() = частка True → множимо на 100 для %
)
# Перетворюємо в DataFrame
age_python_stats = age_python_stats.reset_index()
# Перейменовуємо колонки
age_python_stats.columns = ["Вікова категорія", "Відсоток Python-розробників"]
# Округлюємо відсотки
age_python_stats["Відсоток Python-розробників"] = age_python_stats["Відсоток Python-розробників"].round(2)
# Виводимо результат
age_python_stats.sort_values(by="Відсоток Python-розробників", ascending=False)


,Вікова категорія,Відсоток Python-розробників
0,18-24 years old,40.00
3,45-54 years old,38.63
4,55-64 years old,37.24
1,25-34 years old,36.94
2,35-44 years old,36.72
5,65 years or older,31.63
6,Prefer not to say,31.22


In [24]:
#Завдання 10. Аналіз індустрій серед високооплачуваних віддалених працівників

df_copy = df.copy()
# Видаляємо рядки без зарплати
df_copy = df_copy.dropna(subset=["ConvertedCompYearly"])

# Обчислюємо 75-й перцентиль компенсації
percentile_75 = df_copy["ConvertedCompYearly"].quantile(0.75)
print(f"75-й перцентиль компенсації: {percentile_75:,.0f}")
# Фільтруємо високооплачуваних віддалених працівників
high_paid_remote = df_copy[
    (df_copy["ConvertedCompYearly"] > percentile_75) &
    (df_copy["RemoteWork"] == "Remote")
]
# Підраховуємо популярність індустрій
industry_stats = (
    high_paid_remote["Industry"]
    .value_counts()
    .reset_index()
)
# Перейменовуємо колонки
industry_stats.columns = ["Індустрія", "Кількість респондентів"]
# Виводимо результат
industry_stats

75-й перцентиль компенсації: 120,596


,Індустрія,Кількість респондентів
0,Software Development,1186
1,Fintech,190
2,Healthcare,188
3,Other:,176
4,"Internet, Telecomm or Information Services",138
5,Banking/Financial Services,88
6,Government,78
7,Media & Advertising Services,75
8,Retail and Consumer Services,65
9,"Transportation, or Supply Chain",63
